In [1]:
import sys
import os
import math
import numpy as np

In [2]:
states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0], 
                                  [0.0, 0.9, 0.1, 0.0, 0.0], 
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]]) 
emission_nuc_codes = {'A': 0, 
                      'C': 1, 
                      'G': 2, 
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00], 
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]]) 

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

In [3]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [4]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)

-43.89740030179307


In [5]:
num_states = len(states)
seq_len = len(query_sequence)

state_order = [id2state[i] for i in range(num_states)]
first_nuc = query_sequence[0]

viterbi_value_matrix = np.full((num_states, seq_len), -np.inf, dtype=float)
viterbi_trace_matrix = np.full((num_states, seq_len), -1, dtype=int)

# Initialize first column
for state_name in state_order:
    state_idx = states[state_name]
    trans_prob = state_transition_prob[states["s"]][state_idx]
    emit_prob = emission_probs[state_idx][emission_nuc_codes[first_nuc]]
    
    if trans_prob > 0 and emit_prob > 0:
        viterbi_value_matrix[state_idx, 0] = math.log(trans_prob) + math.log(emit_prob)
    else:
        viterbi_value_matrix[state_idx, 0] = -np.inf
    
    viterbi_trace_matrix[state_idx, 0] = -1

print(viterbi_value_matrix)
print(viterbi_trace_matrix)

[[       -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf]
 [-1.38629436        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf]
 [       -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf]
 [       -inf        -inf      

In [6]:
def calculate_prob_for_a_node(current_state_idx, col_idx):
    current_nuc = query_sequence[col_idx]
    emit_prob = emission_probs[current_state_idx][emission_nuc_codes[current_nuc]]
    
    # If emission probability is 0, return -inf
    if emit_prob == 0:
        return -np.inf, -1
    
    emit_log_prob = math.log(emit_prob)
    max_prob = -np.inf
    max_prev_state_idx = -1
    
    # Iterate through all possible previous states
    for prev_state_idx in range(num_states):
        prev_prob = viterbi_value_matrix[prev_state_idx, col_idx - 1]
        
        # Skip if previous probability is -inf
        if prev_prob == -np.inf:
            continue
        
        trans_prob = state_transition_prob[prev_state_idx][current_state_idx]
        
        # Skip if transition probability is 0
        if trans_prob == 0:
            continue
        
        trans_log_prob = math.log(trans_prob)
        current_prob = prev_prob + trans_log_prob + emit_log_prob
        
        if current_prob > max_prob:
            max_prob = current_prob
            max_prev_state_idx = prev_state_idx
    
    return max_prob, max_prev_state_idx

In [7]:
for col_idx in range(1, seq_len):
    for state_idx in range(num_states):
        max_prob, max_prev_state_idx = calculate_prob_for_a_node(state_idx, col_idx)
        viterbi_value_matrix[state_idx, col_idx] = max_prob
        viterbi_trace_matrix[state_idx, col_idx] = max_prev_state_idx

print("Viterbi Value Matrix:")
print(viterbi_value_matrix)
print("\nViterbi Trace Matrix:")
print(viterbi_trace_matrix)

Viterbi Value Matrix:
[[        -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf]
 [ -1.38629436  -2.87794924  -4.36960411  -5.86125899  -7.35291387
   -8.84456875 -10.33622362 -11.8278785  -13.31953338 -14.81118825
  -16.30284313 -17.79449801 -19.28615288 -20.77780776 -22.26946264
  -23.76111751 -25.25277239 -26.74442727 -28.23608214 -29.72773702
  -31.2193919  -32.71104677 -34.20270165 -35.69435653 -37.1860114
  -38.67766628]
 [        -inf         -inf         -inf         -inf -11.15957636
          -inf -11.19844713         -inf -14.18175689 -18.61785074
  -20.10950562 -21.6011605  -20.14837639         -inf -26.07612513
  -24.62334102 -29.05943488         -inf -29.09830565         -inf
  -35.026

In [8]:
def trace_viterbi_path(viterbi_value_matrix, viterbi_trace_matrix, id2state):
    last_col = viterbi_value_matrix[:, -1]
    current_state_idx = int(np.argmax(last_col))

    path = [id2state[current_state_idx]]
    for col_idx in range(viterbi_value_matrix.shape[1] - 1, 0, -1):
        current_state_idx = viterbi_trace_matrix[current_state_idx, col_idx]
        if current_state_idx == -1:
            break
        path.append(id2state[current_state_idx])

    return "".join(reversed(path))


best_path = trace_viterbi_path(viterbi_value_matrix, viterbi_trace_matrix, id2state)
print(best_path)

EEEEEEEEEEEEEEEEEEEEEEEEEE


In [9]:
get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE","CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)

-40.98025137355685

In [10]:
get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII","CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)

-41.21967768602254